In [1]:
import math
from pathlib import Path
from typing import Tuple
import warnings

from cytoolz import groupby, valfilter
from marslab.geom import transform_angle, sph2cart

from marslab.imgops.imgutils import normalize_range, enhance_color
from marslab.imgops.render import flatten_into_figure
import marslab.parse as mp
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import pdr
from scipy.interpolate import griddata
from scipy.spatial.transform import Rotation
from scipy.optimize import newton

%matplotlib qt

import time

start = time.time()

mpl.rcParams['font.size'] = 32
warnings.simplefilter('ignore', category=RuntimeWarning)

In [2]:
def open_attached(path):
    return pdr.open(path, label_fn=path, skip_existence_check=True)


def get_valid_edges(im: np.ma.MaskedArray):
    if len(im.shape) == 3:
        vy, vx = np.nonzero(np.sum(~im.mask, axis=-1))
    else:
        vy, vx = np.nonzero(~im.mask)
    valid = np.vstack([vy, vx]).T
    v_offset = np.abs(vy - im.shape[0] / 2)
    t_offset = vy
    b_offset = np.abs(vy - im.shape[0])
    h_offset = np.abs(vx - im.shape[1] / 2)
    l_offset = vx
    r_offset = np.abs(vx - im.shape[1])
    return {
        'top': valid[(t_offset + h_offset).argmin()],
        'bottom': valid[(b_offset + h_offset).argmin()],
        'left': valid[(v_offset + l_offset).argmin()],
        'right': valid[(v_offset + r_offset).argmin()]
    }


def get_cahvore(data: pdr.Data, group="GEOMETRIC_CAMERA_MODEL"):
    block = data.metablock(group)
    components = {'reference_frame': block.get('REFERENCE_COORD_SYSTEM_NAME')}
    for comp_ix, id_ in enumerate(block["MODEL_COMPONENT_ID"]):    
        components[id_] = np.array(block[f"MODEL_COMPONENT_{comp_ix + 1}"])
    components['az_fov'] = data.metaget('AZIMUTH_FOV')['value']
    components['el_fov'] = data.metaget('ELEVATION_FOV')['value']
    components['pix_w'] = data.metaget_('LINE_SAMPLES')
    components['pix_h'] = data.metaget_('LINES')
    return components


# something about this is wrong -- the valid edges are too strict, 
# or I'm multiplying the wrong vectors, or something.
def check_in_bounds(nxyz, nav_c, zcam_a):
    edges_ij = get_valid_edges(nxyz)
    nav_edgerays = {
        k: nxyz[v[0], v[1]] - nav_c for k, v in edges_ij.items()
    }
    nz_edge_cross = {
        k: np.cross(edgeray, zcam_a) for k, edgeray in nav_edgerays.items()
    }
    in_bounds_j = (np.sign(nz_edge_cross['top']) != np.sign(nz_edge_cross['bottom']))[1]
    in_bounds_i = (np.sign(nz_edge_cross['left']) != np.sign(nz_edge_cross['right']))[2]
    if in_bounds_i and in_bounds_j:
        return True
    return False


def derive_cahvore_properties(cahvore):
    for ax in ('H', 'V'):
        cahvore[f'{ax}_image'] = (
            cahvore[ax] - np.dot(cahvore['A'], cahvore[ax]) * cahvore['A']
        )
        cahvore[f'{ax}s'] = np.linalg.norm(np.cross(cahvore['A'], cahvore[ax]))
        cahvore[f'{ax}c'] = np.dot(cahvore['A'], cahvore[ax])
    cahvore['dpp_H'] = cahvore['az_fov'] / cahvore['pix_w']
    cahvore['dpp_V'] = cahvore['el_fov'] / cahvore['pix_h']
    for ax in ('H', 'V', 'A'):
        cahvore[f'{ax}u'] = cahvore[ax]/np.linalg.norm(cahvore[ax])
    return cahvore


# rough! and maybe unnecessary?
def ij_to_ray(i, j, cahv):
    i_off = i - cahv['Hc']
    j_off = j - cahv['Vc']
    rot_H = cahv['Hu'] * i_off * cahv['dpp_H']
    rot_V = cahv['Vu'] * j_off * cahv['dpp_V']
    rotate = Rotation.from_rotvec((rot_H, rot_V), degrees=True)
    return np.mean(rotate.apply(cahv['A']), axis=0)


# maybe?
def rough_valid_area(xyzmap, cahvor, slop = 0.8):
    relative_position_vecs = (xyzmap - cahvor['C'])
    rel_pos_mag = np.linalg.norm(relative_position_vecs, axis=2)
    rel_pos_u = np.einsum('ijk,ij->ijk', relative_position_vecs, 1 / rel_pos_mag)
    # plt.imshow(rel_pos_u)  # pretty!
    off_comp_h = np.dot(cahvor['Au'] - rel_pos_u, cahvor['Hu'])
    off_comp_v = np.dot(cahvor['Au'] - rel_pos_u, cahvor['Vu'])
    off_h_deg = np.abs(np.degrees(np.arcsin(off_comp_h)))
    off_v_deg = np.abs(np.degrees(np.arcsin(off_comp_v)))
    return np.nonzero(
        np.logical_and(
            off_h_deg < (cahvor['az_fov'] / (2 - slop)),
            off_v_deg < (cahvor['el_fov'] / (2 - slop))
        )
    )

def prune_xyzmap(xyzmap: np.ma.MaskedArray, cahvore: dict, slop: float=0.8):
    indices = rough_valid_area(xyzmap, cahvore, slop=slop)
    xyz = xyzmap[indices]
    assert not xyz.mask.any(), "invalid pixels have entered the xyzmap"
    return xyz.data, indices


def xyz2ij(xyz, cahvore):
    relative_positions = xyz - cahvore['C']
    omegas = np.dot(relative_positions, cahvore['O'])
    w_omegas = np.einsum('ij,i->ij', np.array([cahvore['O'] for _ in omegas]), omegas)
    lambda3s = relative_positions - w_omegas
    taus = np.einsum('ij,ij->i', lambda3s, lambda3s) / omegas ** 2
    r1, r2, r3 = cahvore['R']
    mus = r1 + r2 * taus + r3 * taus ** 2
    pps = np.einsum('ij,i->ij', lambda3s, mus) + xyz
    pp_cs = pps - cahvore['C']
    ppcs_dot_a = np.dot(pp_cs, cahvore['A'])
    ijvec = np.vstack(
        [
            np.dot(pp_cs, cahvore['H']) / ppcs_dot_a - 1,
            np.dot(pp_cs, cahvore['V']) / ppcs_dot_a - 1,
        ]
    ).T
    return np.round(ijvec).astype(np.int32)


def select_valid_pixels(ij, cahvore):
    validmask = (
        np.all(ij > 0, axis=1)
        & (ij[:, 0] <= zc['pix_w'] - 1) 
        & (ij[:, 1] <= zc['pix_h'] - 1) 
    )
    return np.nonzero(validmask)


def select_and_map_coordinates(xyzmap, target_cahvore):
    # optimization step
    xyz_candidates, indices = prune_xyzmap(xyzmap, target_cahvore)  
    if xyz_candidates.size == 0:
        return {}
    ij_candidates = xyz2ij(xyz_candidates, target_cahvore)
    valid_index = select_valid_pixels(ij_candidates, target_cahvore)
    ij = ij_candidates[valid_index].T
    xyz = xyz_candidates[valid_index].T
    return {
        'i': ij[0], 
        'j': ij[1], 
        'x': xyz[0], 
        'y': xyz[1], 
        'z': xyz[2],
        'si': indices[1][valid_index],
        'sj': indices[0][valid_index]
    }

def make_incidence_map(uvw, img_data):
    sun_vector = sph2cart(*transform_angle('SITE', 'ROVER', 'SOLAR', img_data))
    sun_vector = sun_vector / np.linalg.norm(sun_vector)
    deflection = np.dot(
        np.einsum('ijk,ij->ijk', uvw, 1 / np.linalg.norm(uvw, axis=2)), 
        sun_vector * -1
    )
    return np.degrees(np.arccos(deflection))

def make_rangemap(xyz, origin=(0, 0, 0)):
    return np.linalg.norm(xyz - origin, axis=-1)

def filter_navrec(navrec):
    if len(navrec['coords'].get('i', [])) == 0:
        return False
    return True

def pick_biggest_navrec(nav_recs):
    sizes = [
        len(rec['coords'].get('i', [])) for rec in nav_recs
    ]
    return nav_recs[np.argmax(sizes)]

In [116]:
def sitedrive(path):
    return (mp.site(path.name), mp.drive(path.name))

root = Path('/datascratch/zcam_data/products/')
zsol, nsol = 207, 207
iofdir = Path(root, str(zsol).zfill(4), 'iof')
xyrdir = Path(root, str(nsol).zfill(4), 'xyr')
uvwdir = Path(root, str(nsol).zfill(4), 'uvw')
zsite = groupby(sitedrive, iofdir.iterdir())
nsite = groupby(sitedrive, xyrdir.iterdir())
sd = tuple(set(zsite.keys()).intersection(nsite.keys()))[0]
iofs, xyrs = zsite[sd], nsite[sd]
r2_data = pdr.read([i for i in iofs if i.name.startswith("ZR2")][0])

In [117]:
zc = derive_cahvore_properties(get_cahvore(r2_data))
nav_recs = []
for xyr_file in xyrs:
    xyr = open_attached(xyr_file)
    nc = get_cahvore(xyr)
    nxyz = np.moveaxis(xyr.get_scaled('IMAGE'), 0, 2)
#     plt.figure()
#     plt.imshow(np.sqrt(nxyz[:, :, 0] + 1))
#     plt.title(xyr_file)
    if not nxyz.any():
        continue
    rec = {
        'xyz': nxyz, 
        'fn': xyr.filename, 
        # check_in_bounds is not really working effectively.
        'hit': check_in_bounds(nxyz, nc['C'], zc['A'])
    }   
    nav_recs.append(rec)

In [118]:
for rec in nav_recs:
    rec['coords'] = select_and_map_coordinates(rec['xyz'], zc)
# nav_recs = tuple(filter(filter_navrec, nav_recs))
# want a cutoff for pathological cases that are just going to map sketchy
# data in the far corner of a navcam image -- probably like if it's under 1000 pixels,
# don't use it
rec = pick_biggest_navrec(nav_recs)
del nav_recs

In [119]:
uvw_file = [
    f for f in uvwdir.iterdir()
    if f.name == Path(rec['fn']).name.replace('XYR', 'UVW')
][0]
uvw_data = pdr.read(uvw_file)
nuvw = np.moveaxis(uvw_data.get_scaled('IMAGE'), 0, 2)

In [120]:
for ix, comp in enumerate(('u', 'v', 'w')):
    rec['coords'][comp] = nuvw[rec['coords']['sj'], rec['coords']['si'], ix]

In [121]:
coords = rec['coords']
mesh = np.zeros((*r2_data.IMAGE.shape, 6), np.float32)
for ax_ix, ax in enumerate(('x', 'y', 'z', 'u', 'v', 'w')):
    mesh[coords['j'], coords['i'], ax_ix] = coords[ax]
interp_tracker = np.full(r2_data.IMAGE.shape, 1, np.float32)
ji = np.nonzero(mesh[:, :, 0])
interp_tracker[ji] = 0
missing_ji = np.nonzero(interp_tracker)
gridded = {}
for ax_ix, ax in enumerate(('x', 'y', 'z', 'u', 'v', 'w')):
    values = mesh[:, :, ax_ix][ji]
    if not values[np.isfinite(values)].any():
        continue
    interpolated = griddata(ji, values, missing_ji, method='linear')
    gridarray = np.empty(r2_data.IMAGE.shape, np.float32)
    gridarray[ji] = values
    gridarray[missing_ji] = interpolated
    gridded[ax] = gridarray


In [122]:
xyz = np.dstack([gridded['x'], gridded['y'], gridded['z']])
rangemap = make_rangemap(xyz, zc['C']).astype('f4')
try:
    uvw = np.dstack([gridded['u'], gridded['v'], gridded['w']])
    incidence_map = make_incidence_map(uvw, r2_data).astype('f4')
except KeyError:
    print("sorry, no normals available for this region.")
    uvw, incidence_map = None, None

In [125]:
from marslab.imgops.render import colormapped_plot
image = normalize_range(r2_data.get_scaled('IMAGE'), (0, 1), 1)
alpha = 0.8
image_rgb = np.dstack(
    [image] * 3 + [np.full_like(image, alpha)]
)
fig = colormapped_plot(
    incidence_map,
    layers=[image_rgb], 
#     alpha=0.8, 
    cmap='jet',
    render_colorbar=True,
    drop_mask=False,
#     mask_fill_color=0,
    n_ticks=5
)

In [85]:
rangemesh = np.linalg.norm(mesh[:, :, 0:3], axis=-1)
rangemesh = np.ma.masked_less_equal(rangemesh, 0)

In [86]:
imesh = make_incidence_map(mesh[:, :, 3:6], r2_data)
imesh = np.ma.masked_invalid(imesh)

In [ ]:
plt.imshow(r2_data.IMAGE)

In [ ]:
mpl.style.use('dark_background')
plt.imshow(imesh, interpolation='none', cmap='winter')

In [ ]:
si, sj = rec['coords']['si'], rec['coords']['sj']

In [ ]:
plt.imshow(
    nxyz[sj.min():sj.max(), si.min():si.max(), 1]
)

In [ ]:
nzy, nzx = np.nonzero(rangemesh)

In [ ]:
plt.figure()
plt.imshow(image, cmap='Greys_r')
plt.scatter(
    nzx, nzy, c=(rangemesh[nzy, nzx]), cmap='jet', s=20
)
plt.colorbar()

In [ ]:
from astropy.io import fits

In [ ]:
rangemesh[(nzy, nzx)].shape

In [ ]:
hdus = [
    fits.PrimaryHDU(),
    fits.ImageHDU(xyz, name='xyz'),
    fits.ImageHDU(uvw, name='uvw'),
    fits.ImageHDU(rangemap, name='range'),
    fits.ImageHDU(incidence_map, name='incidence')
]
hdul = fits.HDUList(hdus)

In [ ]:
hdul.writeto('test.fits', overwrite=True)

In [ ]:
plt.figure()
plt.imshow(rangemap, cmap='Greys_r')
plt.colorbar()

In [ ]:
fits.open('test.fits')[1].data

In [ ]:
rangemap = make_rangemap(xyz)
plt.figure()
plt.imshow(rangemap)
plt.colorbar()

In [ ]:
plt.imshow(np.clip(off, 0, 45), cmap='Greys_r')
plt.colorbar()

In [ ]:
xyrs

In [ ]:
plt.imshow(mesh[:, :, 1], interpolation=None, cmap='Greys')

In [ ]:
plt.imshow(r2_data['IMAGE'])

In [ ]:
np.linalg.norm(xyz[545, 869])

In [ ]:
plt.imshow(xyz - xyz[624, 804])

In [ ]:
np.linalg.norm(xyz[1161, 1451] - xyz[1020, 1356])

In [ ]:

# flatten_into_figure([image_rgb, stretched])

In [ ]:
colormapped_plot

In [ ]:
rec

In [ ]:
flatten_into_figure(
    r2_data.IMAGE

In [ ]:
rangemap = np.linalg.norm(xyz - xyz[624, 804], axis=-1)
# np.percentile(rangemap[np.isfinite(rangemap)], (10, 90))
plt.figure()
plt.imshow(rangemap)

In [ ]:
plt.imshow(image)

In [ ]:
plt.figure()
plt.imshow(rangemap)
plt.colorbar()

In [ ]:
plt.contour??

In [ ]:
plt.style.library.keys()

In [ ]:
plt.style.use('dark_background')
rclip = np.clip(rangemap, *np.percentile(rangemap[np.isfinite(rangemap)], (10, 90)))
plt.contour(
    np.arange(rclip.shape[1]),
    np.arange(rclip.shape[0]),
    np.flip(rclip, axis=0),
    levels=30,
    linewidths=5
)
plt.colorbar()

In [ ]:
# # for interpolating multiples...
# # work needs to be done to deal with banding etc.
# flats, meshes = [], []
# for rec in nav_recs:
#     coords = rec['coords']
#     canvas = np.zeros((*r2_data.IMAGE.shape, 3), np.float32)
#     for ax_ix, ax in enumerate(('x', 'y', 'z')):
#         canvas[coords['j'], coords['i'], ax_ix] = coords[ax]
#     meshes.append(canvas)
# pointcounts = np.sum([m[:, :, 0] != 0 for m in meshes], axis=0)
# mesh = np.einsum('ijk,ij->ijk', np.sum(meshes, axis=0), 1 / pointcounts)
# mesh[~np.isfinite(mesh)] = 0
# interp_tracker = np.full(r2_data.IMAGE.shape, 1, np.float32)
# ji = np.nonzero(mesh[:, :, 0])
# interp_tracker[ji] = 0
# missing_ji = np.nonzero(interp_tracker)

In [ ]:
uvw.dtype

In [ ]:
# alpha = 0.6
# xyz_filled = np.dstack(list(gridded.values()))
# xyz_filled = np.ma.masked_invalid(xyz_filled)
# stretched = enhance_color(xyz_filled, (0, 1), (1))
# stretched = np.dstack([stretched, np.full_like(xyz_filled, alpha)])
# image = normalize_range(r2_data.get_scaled('IMAGE'), (0, 1), 1)
# image_rgb = np.dstack([image] * 3)
# flats.append(flatten_into_figure([image_rgb, stretched]))

stop = time.time()

In [ ]:
stop - start

In [ ]:
raise ValueError

In [ ]:
plt.imshow(gridded['z'])

In [ ]:
numpy downample

In [ ]:
fig, ax = plt.subplots(subplot_kw=dict(projection='3d'))
ax.scatter3D(
    xyz[:, :, 0].ravel()[::50],
    xyz[:, :, 1].ravel()[::50],
    xyz[:, :, 2].ravel()[::50],
    c = image.ravel()[::50]
)

In [ ]:
plt.imshow(mesh, interpolation='none')

In [ ]:
plt.close('all')

In [ ]:
ax.plot_surface??

In [ ]:
fig.show()

In [ ]:
mmean = (meshes[4] + meshes[6]) / 2

In [ ]:
distance = np.linalg.norm(xyz_filled, axis=2)

In [ ]:
fig, ax = plt.subplots()

In [ ]:
plt.imshow(off)

In [ ]:
ax.imshow(distance, cmap='cividis')

In [ ]:
r2_data.metaget('SOLAR_ELEVATION')

In [ ]:
np.nanmean(off)

In [ ]:
griddata(ji[:, 0], ji[:, 1])

In [ ]:
ji[:, 0].argsort()

In [ ]:
ji[:, 1].argsort()

In [ ]:
ji[:, 0]

In [ ]:
RegularGridInterpolator((ji[:, 0], ji[:, 1]), xyz[0])

In [ ]:
interp2d?

In [ ]:
ix = 11022
cahvore = zc
relpos, xyz = relative_positions[ix], possible_xyz[ix]
omega = np.dot(relpos, cahvore['O'])
wo = omega * cahvore['O']
lambda3 = relpos - wo
tau = np.dot(lambda3, lambda3) / omega ** 2
r1, r2, r3 = cahvore['R']
mu = r1 + r2 * tau + r3 * tau ** 2
pp = mu * lambda3 + xyz
pp_c = pp - cahvore['C']
ppc_dot_a = np.dot(pp_c, cahvore['A'])
res = {
    'ij': (
        np.dot(pp_c, cahvore['H']) / ppc_dot_a - 1,
        np.dot(pp_c, cahvore['V']) / ppc_dot_a - 1
    ), 
    'xyz': xyz
}

In [ ]:
## i think this works, though!


# what do a lot of these variables stand for? incomprehensible
def xyz2ij(xyz, relpos, cahvore):
    omega = np.dot(relpos, cahvore['O'])
    wo = omega * cahvore['O']
    lambda3 = relpos - wo
    tau = np.dot(lambda3, lambda3) / omega ** 2
    r1, r2, r3 = cahvore['R']
    mu = r1 + r2 * tau + r3 * tau ** 2
    pp = mu * lambda3 + xyz
    pp_c = pp - cahvore['C']
    ppc_dot_a = np.dot(pp_c, cahvore['A'])
    return {
        'ij': (
            np.dot(pp_c, cahvore['H']) / ppc_dot_a - 1,
            np.dot(pp_c, cahvore['V']) / ppc_dot_a - 1
        ), 
        'xyz': xyz
    }

In [ ]:
omega = np.dot(relative_positions, zc['O'])

In [ ]:
located = [xyz2ij(xyz, relpos, zc) for xyz, relpos in zip(possible_xyz, relative_positions)]

In [ ]:
valid_ij = []
for coord in ij:
    if not 0 <= coord[0] <= 1648:
        continue
    if not 0 <= coord[1] <= 1200:
        continue
    valid_ij.append(coord)

In [ ]:
len(valid_ij)

In [ ]:
ij[-10]

In [ ]:
r2_data.IMAGE.shape

In [ ]:
# maybe unnecssary?
zcam_edges = get_valid_edges(r2_data.get_scaled('IMAGE'))
zcam_edge_rays = {
    k: ij_to_ray(yx[1], yx[0], zc)
    for k, yx in zcam_edges.items()
}

In [ ]:
# not sure this cross product is useful
# pos_cross_a_mag = np.linalg.norm(pos_cross_a, axis=2)
# pos_cross_a_u = np.einsum('ijk,ij->ijk', pos_cross_a, 1 / pos_cross_a_mag)
# i think taking these dot products is incoherent...
# pos_cross_a_dot_h = np.dot(pos_cross_a_u, zc['Hu'])
# pos_cross_a_dot_v = np.dot(pos_cross_a_u, zc['Vu'])
# not sure these dot products are useful
# pos_dot_h = np.dot(rel_pos_u, zc['Hu'])
# pos_dot_v = np.dot(rel_pos_u, zc['Vu'])

In [ ]:
plt.imshow(rough_valid_area)
bigmask = np.logical_or(nxyz.mask, np.dstack([~rough_valid_area]*3))
nxyz_v = nxyz.copy()
nxyz_v.mask = bigmask
nxyz_v[nxyz_v.mask] = 0
plt.figure()
plt.imshow(nxyz_v)

In [ ]:
# xyr = open_attached(xyrs[1])
# nav_cahvore = get_cahvore(xyr)
# (xyr.metaget('STEREO_PRODUCT_ID'), xyr.metaget('SOURCE_PRODUCT_ID'), xyr.metaget('INPUT_PRODUCT_ID'))
# nav_input = pdr.read(
#     '/datascratch/zcam_data/products/0092/nav/NLFC0092_0675117098_000RASLN0040136NCAM00699_0A00LLJ01.IMG'
# )
# navim = normalize_range(np.moveaxis(nav_input.IMAGE, 0, 2), (0, 1), 1)
# plt.figure()
# plt.imshow(navim)
# plt.figure()
# plt.imshow(np.sqrt(nxyz[:, :, 0] + 1))

In [ ]:
from scipy.spatial.transform import Rotation

In [ ]:
Rotation.from_rotvec()

In [ ]:
plt.imshow(np.sqrt(nxyz[:, :, 0]))

see: https://pds-imaging.jpl.nasa.gov/data/mer/spirit/mer2no_0xxx/document/geometric_cm.txt

so what is this ij_to_ray?

in CAHV model strictly: 

```
Given a 3D point P, the image coordinates may be computed as

             (P-C)*H         (P-C)*V 
         X = -------     Y = -------        
             (P-C)*A         (P-C)*A 

         (Note: * = vector dot product)
```

the dot product is of course not injective, so these equations cannot simply be inverted.

note, though, that A, H, V are _close_ to mutually orthogonal even given the ORE parameters...


```
A projection of this vector into the image plane,
called H', shows how the horizontal image dimension (the rows)
are oriented in 3D.The magnitude of this projection is, for an
ideal thin lens, the distance between the lens center and the
image plane, as measured in horizontal pixels; this value is
called the Horizontal Scale, and is often written as Hs.
(Contrary to what has been suggested elsewhere, Hs is not the focal length.)
[and then the same is true for V/vertical pixels]

```

```
H' = H-(A*H)A     Hs = |AxH|    Hc = A*H
```

```
Note that H' and V' are not necessarily exactly perpendicular to each
other. The angle between them, theta, is a measure of the spatial
relationship of the rows and columns. They are usually very close to
perpendicular, but have sometimes been observed to fall more than a
standard deviation away from that ideal.

                             (   VxH*A   ) 
               Theta = arctan(-----------) 
                             ((AxV)*(AxH))
```

In [ ]:
zcam_cahvore = derive_cahvore_properties(zcam_cahvore)

In [ ]:
zcam_cahvore

In [ ]:
zcam_cahvore['H_image'] / zcam_cahvore['Hs']

In [ ]:
np.linalg.norm(zcam_cahvore['V'])

In [ ]:
plt.imshow(r2.get_scaled('IMAGE'))

In [ ]:
# so i think we're talking about like y=2313, x=2680 for center of zcam image in xyr image coords

In [ ]:
xyz_rov = nxyz[2313, 2680].data

In [ ]:
xyz_rov

In [ ]:
plt.figure()
plt.imshow(np.clip(nxyz[:, :, 1], 0.2, 0.8))

In [ ]:
zcam_cahvore['C'], zcam_cahvore['A']

In [ ]:
test_position = zcam_cahvore['C'] + zcam_cahvore['A'] * 4
# zcam_cahvore['H'], zcam_cahvore['V']

In [ ]:
h_hat = zcam_cahvore['H'] / np.linalg.norm(zcam_cahvore['H'])
v_hat = zcam_cahvore['V'] / np.linalg.norm(zcam_cahvore['V'])
a_hat = zcam_cahvore['A'] / np.linalg.norm(zcam_cahvore['A'])

In [ ]:
az_fov = r2

In [ ]:
  AZIMUTH_FOV                     = 6.36042 <deg>
  ELEVATION_FOV                   = 4.63709 <deg>
  BAYER_METHOD                    = IDENTITY
  CFA_TYPE                        = BAYER_RGGB
  CFA_VENUE                       = ONBOARD
  DETECTOR_FIRST_LINE             = 1
  DETECTOR_FIRST_LINE_SAMPLE      = 1
  DETECTOR_LINES                  = 1200
  DETECTOR_LINE_SAMPLES           = 1648

In [ ]:
np.degrees(np.arccos(np.dot(a_hat, v_hat)))

In [ ]:
len(np.nonzero(~nxyz.mask)[0]) * 200

In [ ]:
i, j

In [ ]:
# # this should be cool but in fact will need more work to be cool
# from marslab.imgops.imgutils import enhance_color
# im = np.moveaxis(im, 0, 2)
# display = enhance_color(im, (0, 1), 1)
# plt.imshow(display)